# OpenAI usage — Knight Vision

**Full account usage** (all API calls, including game annotation) requires an admin key:

1. [Create admin key](https://platform.openai.com/settings/organization/admin-keys) with **`api.usage.read`**
2. Add to repo-root `.env`: `OPENAI_ADMIN_KEY=sk-admin-...`
3. Re-run the cell below

Your regular `OPENAI_API_KEY` can call the API but **cannot** read billing/usage (OpenAI returns 403).

Without an admin key: use the [usage dashboard](https://platform.openai.com/usage), or see the local Knight Vision log at the bottom (only calls made after we added logging).

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "backend").is_dir():
    REPO_ROOT = REPO_ROOT.parent
ENV_FILE = REPO_ROOT / ".env"
load_dotenv(ENV_FILE, override=True)
sys.path.insert(0, str(REPO_ROOT / "backend"))

from config import openai_admin_key, openai_api_key
from openai_usage_log import COLUMNS, LOG_PATH, load_all
from openai_usage_query import OpenAIUsageError, fetch_completions_usage, fetch_costs

DAYS = 30
REAL_SOURCES = {"generate_lessons", "api"}

print("Env file:", ENV_FILE, "(exists:", ENV_FILE.is_file(), ")")
print("OPENAI_API_KEY:", "yes" if openai_api_key() else "no")
print("OPENAI_ADMIN_KEY:", "yes" if openai_admin_key() else "no")
print("Local log:", LOG_PATH, "(exists:", LOG_PATH.is_file(), ")")
print()


def openai_cost_summary(costs: pd.DataFrame) -> pd.DataFrame:
    if costs.empty:
        return pd.DataFrame(
            [{"first_day": None, "last_day": None, "days_with_spend": 0, "total_cost_usd": 0.0}]
        )
    by_day = costs.groupby("date", as_index=False)["cost_usd"].sum()
    spend_days = by_day[by_day["cost_usd"] > 0]
    return pd.DataFrame(
        [
            {
                "first_day": spend_days["date"].min() if len(spend_days) else None,
                "last_day": spend_days["date"].max() if len(spend_days) else None,
                "days_with_spend": len(spend_days),
                "total_cost_usd": round(float(costs["cost_usd"].fillna(0).sum()), 4),
            }
        ]
    )


def local_calls_df(*, real_only: bool = False) -> pd.DataFrame:
    rows = load_all()
    if not rows:
        return pd.DataFrame(columns=COLUMNS)
    df = pd.DataFrame(rows)[COLUMNS].sort_values("event_id")
    if real_only:
        df = df[df["source"].isin(REAL_SOURCES)]
    return df.reset_index(drop=True)


def local_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(
            [{"first_event": None, "last_event": None, "events": 0, "total_cost_usd": 0.0}]
        )
    def label(row: pd.Series) -> str:
        return f"#{int(row['event_id'])} [{row['source']}] {row['label']}"
    return pd.DataFrame(
        [
            {
                "first_event": label(df.iloc[0]),
                "last_event": label(df.iloc[-1]),
                "events": len(df),
                "total_cost_usd": round(float(df["cost_usd"].sum()), 6),
            }
        ]
    )


# ── OpenAI account usage (needs admin key, or project key with usage scope) ──
openai_ok = False
try:
    cost_rows = fetch_costs(days=DAYS, group_by=["line_item"])
    usage_rows = fetch_completions_usage(days=DAYS, group_by=["model"])
    openai_ok = True
except OpenAIUsageError as e:
    print("OpenAI account API:", e)
    print("→ Full history: https://platform.openai.com/usage\n")
else:
    costs = pd.DataFrame(cost_rows)
    usage = pd.DataFrame(usage_rows)
    print(f"OpenAI account — last {DAYS} days")
    display(openai_cost_summary(costs))
    if not costs.empty:
        print("Daily spend by line item")
        display(
            costs.groupby(["date", "line_item"], as_index=False)["cost_usd"]
            .sum()
            .sort_values(["date", "line_item"])
        )
    if not usage.empty:
        print("Completion tokens by model (daily)")
        display(
            usage.groupby(["date", "model"], as_index=False)
            .agg({"requests": "sum", "input_tokens": "sum", "output_tokens": "sum"})
            .sort_values(["date", "model"])
        )
    print()

# ── Local Knight Vision log (per-call, since logging was added) ──
local = local_calls_df(real_only=False)
local_app = local_calls_df(real_only=True)
print("Local Knight Vision log (all sources)")
display(local_summary(local))
if local.empty:
    print("(empty — app/generate_lessons calls will appear here after logging was added)")
else:
    display(local)
if not local_app.empty:
    print("\nApp-only (generate_lessons + api, excluding notebook demos)")
    display(local_summary(local_app))
    display(local_app)